In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

ROOT = Path.cwd()
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from song_cleaning.key_decoding import (
    build_circular_distance_matrix,
    decode_argmax,
    decode_min_expected_distance,
    expected_distance_for_choice,
    max_class_confidence,
)
from song_cleaning.key_metrics import compute_key_metrics

DATA_PATH = ROOT / "liveness_final.parquet"
FEATURE_POOL = [
    "valence",
    "loudness",
    "danceability",
    "energy",
    "tempo",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "mode",
    "valence_missing",
    "loudness_missing",
    "danceability_missing",
    "energy_missing",
    "tempo_missing",
    "acousticness_missing",
    "instrumentalness_missing",
    "liveness_missing",
    "speechiness_missing",
    "key_missing",
    "mode_missing",
]
MODEL_PARAMS = {
    "objective": "multi:softprob",
    "num_class": 12,
    "eval_metric": "mlogloss",
    "n_estimators": 600,
    "max_depth": 6,
    "learning_rate": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "gamma": 0.0,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}


In [2]:
columns_to_load = sorted(set(FEATURE_POOL + ["key", "track_id", "artist_name", "track_name"]))
df = pd.read_parquet(DATA_PATH, columns=columns_to_load)
float64_columns = df.select_dtypes(include=["float64"]).columns
df[float64_columns] = df[float64_columns].astype("float32")
df[["key"]].isna().sum()


key    446161
dtype: int64

In [3]:
observed_df = df[df["key"].notna()].copy()
missing_df = df[df["key"].isna()].copy()

X_observed = observed_df[FEATURE_POOL]
y_observed = observed_df["key"].astype("int32")
X_missing = missing_df[FEATURE_POOL]

X_train, X_test, y_train, y_test = train_test_split(
    X_observed,
    y_observed,
    test_size=0.2,
    random_state=42,
    stratify=y_observed,
)

len(X_train), len(X_test), len(X_missing)


(2342990, 585748, 446161)

In [4]:
model = XGBClassifier(**MODEL_PARAMS)
model.fit(X_train, y_train)

test_probabilities = model.predict_proba(X_test)
distance_matrix = build_circular_distance_matrix()

argmax_predictions = decode_argmax(test_probabilities)
distance_aware_predictions = decode_min_expected_distance(test_probabilities, distance_matrix)

argmax_metrics = compute_key_metrics(y_test.to_numpy(), argmax_predictions)
distance_aware_metrics = compute_key_metrics(y_test.to_numpy(), distance_aware_predictions)

comparison = pd.DataFrame([
    {"decoder": "argmax", **argmax_metrics},
    {"decoder": "min_expected_distance", **distance_aware_metrics},
])
comparison


,decoder,accuracy,f1_macro,mean_key_distance,median_key_distance,within_1_semitone,within_2_semitones
0,argmax,0.167881,0.126579,2.834231,3.0,0.300783,0.470504
1,min_expected_distance,0.148508,0.096641,2.771931,3.0,0.289539,0.473487


In [5]:
final_model = XGBClassifier(**MODEL_PARAMS)
final_model.fit(X_observed, y_observed)

missing_probabilities = final_model.predict_proba(X_missing)
distance_matrix = build_circular_distance_matrix()

imputed_key_argmax = decode_argmax(missing_probabilities)
imputed_key_distance = decode_min_expected_distance(missing_probabilities, distance_matrix)

imputed_rows = missing_df[["track_id", "artist_name", "track_name"]].copy()
imputed_rows["key_argmax"] = imputed_key_argmax
imputed_rows["key_distance_aware"] = imputed_key_distance
imputed_rows["max_confidence"] = max_class_confidence(missing_probabilities)
imputed_rows["expected_distance"] = expected_distance_for_choice(
    missing_probabilities,
    imputed_key_distance,
    distance_matrix,
)
imputed_rows.head()


,track_id,artist_name,track_name,key_argmax,key_distance_aware,max_confidence,expected_distance
3,NaN,yungmanny,waldo,1,1,0.338606,2.333585
9,NaN,creature feature,bound and gagged,9,9,0.182167,2.689606
20,NaN,three 6 mafia,south memphis bitch,0,0,0.190076,2.636987
23,NaN,roxy music,for your pleasure,0,0,0.154125,2.817933
28,NaN,stick figure,angels among us,0,0,0.179554,2.696455
